***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [6. 成像中的去卷积](6_0_introduction.ipynb)
    * 上一节： [6.0 成像中的去卷积](6_0_introduction.ipynb)
    * 下一节： [6.2 CLEAN](6_2_clean.ipynb)

***


## 6.1 天空模型<a id='deconv:sec:skymodels'></a>

去卷积的对象不是一张普通照片，而是一个由不完整 Fourier 采样决定的反问题。干涉阵列测得的是天空亮度分布在若干空间频率点上的可见度；把这些采样点加权后反变换，得到的是脏图像。若暂时忽略宽视场项和方向相关效应，脏图像可以写成

$$
I_D(l,m)=B_D(l,m) \ast I(l,m) + n_D(l,m),
$$

其中 $I(l,m)$ 是真实天空亮度，$B_D(l,m)$ 是由采样函数决定的脏波束或 PSF，$n_D$ 是被同一成像算子带到图像域的噪声与残差。这里的 $\ast$ 表示卷积。由于采样函数中存在大量没有被观测到的空间频率，许多不同的天空亮度分布都可以给出相同或近似相同的观测数据。因此，去卷积不可能只是“除以 PSF”。它必须引入天空模型，也就是对天空亮度可允许形态的一种参数化假设。

天空模型可以很简单，例如若干点源；也可以很复杂，例如多尺度分量、谱指数项、偏振分量、shapelet 或其他函数基。本章先从点源模型开始，因为这是理解 `CLEAN` 的最短路径。点源模型并不声称所有射电源都是真正的数学点，而是在阵列分辨率有限时，把未解析或近似未解析的结构表示为一组 delta 函数分量。


![天空模型与脏波束](figures/sky_model_and_psf.png)

**图 6.1.1** 左图给出由若干紧致分量组成的合成天空模型，右图给出一个具有主瓣和旁瓣结构的脏波束。脏波束由阵列的空间频率采样决定，而不是由天空本身决定。


从模型到脏图像的正向过程很清楚：给定一个模型 $I_M$，用同一个脏波束做卷积，就得到该模型在当前阵列和成像权重下应产生的脏图响应，

$$
I_{M,D}(l,m)=B_D(l,m) \ast I_M(l,m).
$$

观测脏图像与这个模型脏响应之间的差，就是当前模型没有解释的部分，

$$
R_D(l,m)=I_D(l,m)-B_D(l,m) \ast I_M(l,m).
$$

这一定义非常重要。`CLEAN` 的每一步都可以理解为在残图 $R_D$ 中寻找一个尚未解释的显著结构，把它加入模型，再用 PSF 从残图中减掉相应响应。残图不是“去卷积完成后的废料”，而是模型是否充分、停止准则是否合理、以及图像能否进入科学测量的核心诊断量。


![脏图像的卷积关系](figures/dirty_image_as_convolution.png)

**图 6.1.2** 左图是天空模型与脏波束卷积后的响应；中图在此基础上加入了残余结构和噪声，形成脏图像；右图显示脏图像中没有被当前模型解释的残差项。真实数据中的残差还会包含校准误差、未建模展源结构和方向相关效应。


### 6.1.1 点源假设

点源模型的数学形式最简单。位于 $(l_0,m_0)$、流量密度为 $C(
u)$ 的单个点源可以写成

$$
I_
u(l,m)=C(
u)\,\delta(l-l_0)\,\delta(m-m_0).
$$

把它代入二维 Fourier 变换，得到

$$
\begin{aligned}
V_
u(u,v)
&=\int\int I_
u(l,m)\,e^{-2\pi i(ul+vm)}\,dl\,dm \\
&=C(
u)\int\int \delta(l-l_0)\,\delta(m-m_0)\,e^{-2\pi i(ul+vm)}\,dl\,dm \\
&=C(
u)e^{-2\pi i(ul_0+vm_0)} .
\end{aligned}
$$

这个结果说明两件事。第一，点源的可见度振幅不随基线长度下降；所有基线都测到同一个总流量 $C(
u)$。第二，点源位置只体现在相位斜率中；源离相位中心越远，随 $(u,v)$ 变化的相位越快。多个点源的模型只是这些相位因子的线性叠加，

$$
V_{M,\nu}(u,v)=\sum_k C_k(
u)e^{-2\pi i(ul_k+vm_k)}.
$$

这就是点源天空模型能够直接连接图像域和可见度域的原因。


| Source ID | $l$ | $m$ | $S_0$ (Jy) | $\alpha$ |
|:--|--:|--:|--:|--:|
| S1 | -0.42 |  0.28 | 1.00 | -0.70 |
| S2 |  0.18 | -0.18 | 0.72 | -0.10 |
| S3 |  0.48 |  0.34 | 0.42 | -1.10 |

**表 6.1.1** 一个最小天空模型可以只包含源位置、参考频率处的流量密度 $S_0$ 和谱指数 $\alpha$。若采用幂律谱，频率依赖可写成 $S(\nu)=S_0(\nu/\nu_0)^\alpha$。宽带成像时，谱指数不再只是目录附属量，而会进入多频率去卷积模型。


模型分量并不会直接作为最终图像发布。标准 `CLEAN` 流程通常先得到一组 delta 函数分量，然后用一个称为 clean beam 的理想化主瓣对这些分量进行卷积，再把残图加回去，形成恢复图像，

$$
I_R(l,m)=B_C(l,m) \ast I_M(l,m)+R_D(l,m).
$$

这里 $B_C$ 通常取为拟合脏波束主瓣得到的二维高斯。这样做的目的，是让恢复图像具有明确、稳定、易于引用的角分辨率，同时保留残图中尚未建模的噪声和弱结构。若直接显示 delta 函数模型，图像会显得分辨率无限高；若直接显示脏图像，旁瓣结构又会把源的形态和阵列响应混在一起。恢复图像正是在这两者之间建立可解释的折中。


![由模型生成恢复图像](figures/restored_image_from_model.png)

**图 6.1.3** 恢复图像由两部分相加而成：天空模型与 clean beam 卷积后的平滑分量，以及脏残图。clean beam 只代表成像结果中被声明的有效分辨率；残图仍然保留真实噪声、未建模结构和可能的系统误差。


### 6.1.2 解析源

“点源”是相对于阵列分辨率而言的观测概念，而不是天体的绝对属性。若源的角尺度远小于合成波束主瓣宽度，它在当前阵列上不可解析，点源近似通常足够好。若源的角尺度与合成波束相当或更大，它会被解析，长基线测得的可见度振幅会低于短基线。对展源而言，delta 函数模型虽然仍可用大量点分量进行逼近，但这种逼近会变得低效，并且容易把噪声和旁瓣结构吸收到模型里。

以圆对称高斯源为例，若图像域亮度为

$$
I(l,m)=\frac{S}{2\pi\sigma^2}\exp\left[-\frac{l^2+m^2}{2\sigma^2}\right],
$$

则其可见度振幅为

$$
|V(\rho)|=S\exp\left(-2\pi^2\sigma^2\rho^2\right),\qquad \rho=\sqrt{u^2+v^2}.
$$

这个公式把“源越大，可见度下降越快”说得非常直接。角尺度 $\sigma$ 越大，Fourier 域中的响应越窄，短基线对总流量越重要；角尺度越小，响应越接近常数，点源近似越好。因此，解析源不仅是图像域里“看起来变宽”的问题，也是可见度域中不同基线对同一源测得不同振幅的问题。


![展源尺度与可见度振幅](figures/resolved_gaussian_visibility.png)

**图 6.1.4** 上排展示总流量相同但角尺度不同的高斯源。下排给出其归一化可见度振幅随基线长度的变化。源越大，振幅越快下降；最紧致的源在可观测基线范围内更接近点源。


点源模型、多尺度模型和高斯模型之间并不存在绝对优劣。点源模型适合紧致源和稀疏场，计算简单，残差解释也最直观；多尺度模型更适合弥散结构，可以减少把展源拆成大量点分量时产生的条纹和偏差；参数化高斯模型常用于源搜索和目录构建，因为它能直接给出位置、峰值、积分流量、主轴、次轴和位置角。后续几节讨论的不同 `CLEAN` 变体，本质上就是在这些天空先验、计算代价和残图质量之间选择不同的折中。


***

* 下一节： [6.2 CLEAN](6_2_clean.ipynb)
